In [ ]:
!apt-get update
!apt-get install tesseract-ocr
!pip install pytesseract

In [ ]:
import cv2
import pytesseract
import numpy as np
import sys
import re
from collections import Counter
from google.colab import files
from google.colab.patches import cv2_imshow

def parse_plate(text):
    # Bersihkan karakter aneh
    text = re.sub(r'[^A-Z0-9]', '', text.upper())

    # Cari pola plat nomor standar Indonesia
    match = re.search(r"([A-Z]{1,3})(\d{1,4})([A-Z0-9]{0,3})", text)
    if not match:
        return None

    matched_str = match.group(0)
    if len(matched_str) < len(text) * 0.4:
        return None

    depan = match.group(1).replace('8', 'B').replace('0', 'D')

    if depan[0] in ['O', 'I', 'Q']:
        return None

    if len(depan) == 3 and depan[-1] in ['O', 'Q', 'D']:
        depan = depan[:-1]

    if len(depan) == 2 and depan[-1] in ['O', 'Q']:
        depan = depan[:-1]

    if len(depan) > 2:
        return None

    tengah = match.group(2).lstrip('0')
    if not tengah:
        return None

    if len(tengah) > 4:
        tengah = tengah[:4]

    belakang = match.group(3)

    # Ganti hanya karakter yang paling sering salah dibaca oleh Tesseract untuk bagian belakang
    belakang = belakang.replace('0', 'O').replace('2', 'Z').replace('5', 'S')

    # KUNCI UTAMA: Bagian belakang plat nomor Indonesia HARUS berupa huruf sepenuhnya!
    if belakang and not belakang.isalpha():
        return None

    if len(belakang) > 3:
        belakang = belakang[-3:]

    if 4 <= len(depan + tengah + belakang) <= 9:
        return f"{depan} {tengah} {belakang}".strip()
    return None

def calculate_score(text, count):
    pure_len = len(text.replace(' ', ''))
    return (pure_len * 10) + count

# ==============================================================================
# ALGORITMA PENYAPU BERSIH NOISE & PENYARING STRUKTUR PLAT
# ==============================================================================
def remove_noise(binary_img):
    # 1. Hapus Glare Blob dengan opening sedang
    kernel_glare = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    glare_blob = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel_glare)
    text_only = cv2.subtract(binary_img, glare_blob)

    # 2. Temukan kontur menggunakan RETR_LIST agar teks bertingkat ikut terbaca
    contours, _ = cv2.findContours(text_only, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return text_only

    h_img, w_img = text_only.shape
    img_area = h_img * w_img
    valid_cnts = []

    # Filter kontur berdasarkan struktur geometri teks plat nomor
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if h == 0 or w == 0:
            continue
        # A. Tolak kontur yang lebarnya > 85% layar (Pasti frame plat luar!)
        if w > 0.85 * w_img:
            continue
        # B. Tolak kontur raksasa berdasarkan area
        if w * h > 0.5 * img_area:
            continue
        # C. Tolak noise super kecil
        if w < 2 or h < 8:
            continue
        # D. Tolak bracket plat / garis vertikal tipis (Rasio lebar/tinggi sangat kecil)
        if w / h < 0.12:
            continue
        # E. Tolak garis horizontal tipis (seperti garis pembatas plat)
        if w / h > 2.0:
            continue

        valid_cnts.append(cnt)

    if not valid_cnts:
        return np.zeros_like(text_only)

    # Hitung tinggi referensi huruf utama dari 3 kontur tertinggi
    heights = [cv2.boundingRect(cnt)[3] for cnt in valid_cnts]
    heights.sort(reverse=True)
    ref_h = sum(heights[:3]) / min(len(heights), 3)

    # Hitung median koordinat Y untuk meluruskan teks utama dan membuang tanggal pajak/baut luar baris
    y_centers = []
    for cnt in valid_cnts:
        x, y, w, h = cv2.boundingRect(cnt)
        if h > 0.5 * ref_h:
            y_centers.append(y + h/2)

    if y_centers:
        median_y = np.median(y_centers)
    else:
        median_y = h_img / 2

    # Gambar ulang kontur yang tingginya proporsional DAN sejajar horizontal dengan teks utama
    mask = np.zeros_like(text_only)
    for cnt in valid_cnts:
        x, y, w, h = cv2.boundingRect(cnt)
        cy = y + h/2
        if 0.45 * ref_h < h < 1.5 * ref_h:
            if abs(cy - median_y) < 0.8 * ref_h:
                cv2.drawContours(mask, [cnt], -1, 255, -1)

    cleaned = cv2.bitwise_and(text_only, mask)
    return cleaned

# =========================================
# 1. AKUISISI CITRA VIA UPLOAD COLAB
# =========================================
print("=========================================")
print("  PROGRAM DETEKSI PLAT NOMOR (OCR) ")
print("  v22.0 - GOOGLE COLAB EDITION")
print("=========================================")
print("Silakan pilih gambar plat nomor dari perangkat lu:")

# Memanggil widget upload Colab
uploaded = files.upload()

if not uploaded:
    print("\n[ERROR] Batal upload gambar! Silakan run ulang cell ini.")
    sys.exit()

# Mengambil nama file dari dictionary gambar yang baru diupload
nama_file = list(uploaded.keys())[0]
img = cv2.imread(nama_file)

if img is None:
    print(f"\n[ERROR] Waduh, file '{nama_file}' rusak atau gagal dibaca!")
    sys.exit()

h, w = img.shape[:2]
target_w = 600
scale = target_w / w
img_prep = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC)

# =========================================
# 2. PRE-PROCESSING: ENSEMBLE PREPARATION
# =========================================
print("\nMengekstrak citra dengan Multi-Scale Adaptive Thresholding...")
gray = cv2.cvtColor(img_prep, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5,5), 0)

preprocessed_images = []

thresh_adaptive31 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 10)
preprocessed_images.append(("Adaptive B31", thresh_adaptive31))

thresh_adaptive51 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 51, 10)
preprocessed_images.append(("Adaptive B51", thresh_adaptive51))

# =========================================
# 3. OCR DENGAN ENSEMBLE VOTING SYSTEM
# =========================================
configs = [
    r'--oem 3 --psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
    r'--oem 3 --psm 11 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
]

candidates = []

print("Membersihkan noise dan melakukan OCR (Voting System Aktif)...")

for nama_strategi, citra_prep in preprocessed_images:

    # Tes polaritas ganda (Normal & Inverted)
    citra_inverted = cv2.bitwise_not(citra_prep)

    for img_variant in [citra_prep, citra_inverted]:
        # Bersihkan noise dengan algoritma terpadu baru kita
        cleaned = remove_noise(img_variant)

        # Tesseract membaca Teks Hitam di Latar Putih dengan sangat baik.
        final_for_ocr = cv2.bitwise_not(cleaned)

        # Skala padat untuk akurasi optimal character thickness
        for scale_factor in [1.0, 0.8, 0.7, 0.6, 0.5, 0.4]:
            if scale_factor != 1.0:
                img_scaled = cv2.resize(final_for_ocr, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_AREA)
            else:
                img_scaled = final_for_ocr

            img_ocr = cv2.copyMakeBorder(img_scaled, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=[255, 255, 255])

            for config in configs:
                teks = pytesseract.image_to_string(img_ocr, config=config)
                for baris in teks.split('\n'):
                    parsed = parse_plate(baris)
                    if parsed:
                        candidates.append((parsed, nama_strategi, img_ocr))

# =========================================
# 4. HASIL AKHIR
# =========================================
if candidates:
    counts = Counter([c[0] for c in candidates])

    best_text = sorted(counts.keys(), key=lambda x: calculate_score(x, counts[x]), reverse=True)[0]

    for c in candidates:
        if c[0] == best_text:
             hasil_final = best_text
             gambar_terbaik = c[2]
             nama_strategi_berhasil = c[1]
             break

    print("\n==============================")
    print("HASIL PEMBACAAN PLAT NOMOR:")
    print(f"[{hasil_final}]")
    print(f"Metode Terbaik: {nama_strategi_berhasil}")
    score_pemenang = calculate_score(best_text, counts[best_text])
    print(f"(Voting System: Terbaca {counts[best_text]} kali | Total Skor Kepercayaan: {score_pemenang})")
    print("==============================\n")

    print("1. Plat Asli:")
    cv2_imshow(img)
    print("\n2. Hasil Terbaik Pilihan AI (Pre-processed):")
    cv2_imshow(gambar_terbaik)
else:
    print("\n==============================")
    print("HASIL PEMBACAAN PLAT NOMOR:")
    print("[TIDAK DAPAT MEMBACA PLAT SECARA VALID]")
    print("==============================")
    print("Saran: Kualitas gambar mungkin terlalu buram atau resolusi tidak ideal.\n")

    print("Plat Asli:")
    cv2_imshow(img)